In [5]:
import pandas as pd
import numpy as np

%pip install folium
import folium
from folium.plugins import MarkerCluster

# Load dataset bersih dari folder data/
df = pd.read_csv("../data/kost_putri_cleaned.csv")

print(f"Total data kos: {len(df)} baris")
df[['Nama Kos', 'Alamat', 'Link Google Maps', 'harga_min']].head()


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: C:\Users\Candra\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   ---------------------------------------- 3/3 [folium]

Note: you may need to restart the kernel to use updated packages.
Total data kos: 63 baris


,Nama Kos,Alamat,Link Google Maps,harga_min
0,Kost Putri Asri,"Puluhdadi No.416, Ngropoh, Caturtunggal, Kec. ...",Kos Putri Asri,750000.0
1,Kost Putri Pak Seno,"Jl. Seturan No.1, Kledokan, Caturtunggal, Kec....",Kos Putri Pak Seno,800000.0
2,Kost Perum APH Seturan A.24,"Perum APH Seturan Blok A.24 Kledokan, Caturtun...",Kledokan,600000.0
3,Graha Seturan Kost Putri,"Jln. Seturan Raya, Kledokan, Caturtunggal, Dep...",https://maps.app.goo.gl/279if6Z1PG6ZnCK97,850000.0
4,Kost Diajeng,"Puluhdadi, Kledokan, Caturtunggal, Kec. Depok,...",Kost Putri Muslim Diajeng,900000.0


In [6]:
# Koordinat titik tengah area-area utama di sekitar Seturan/Depok Sleman
AREA_COORDINATES = {
    'kledokan': (-7.7765, 110.4095),
    'puluhdadi': (-7.7715, 110.4080),
    'ngropoh': (-7.7665, 110.4070),
    'condongcatur': (-7.7630, 110.4050),
    'dabag': (-7.7680, 110.3995),
    'seturan': (-7.7735, 110.4088)
}

# Koordinat default (Pusat Seturan)
DEFAULT_CENTER = (-7.7745, 110.4085)

def assign_coordinates(row, seed_offset=0):
    """
    Menentukan estimasi latitude dan longitude berdasarkan pencocokan area alamat.
    Diberikan jitter/offset kecil yang deterministik agar pin tidak bertumpuk di satu titik persis.
    """
    text = (str(row.get('Alamat', '')) + " " + str(row.get('Nama Kos', ''))).lower()
    
    base_lat, base_lng = DEFAULT_CENTER
    for area, (lat, lng) in AREA_COORDINATES.items():
        if area in text:
            base_lat, base_lng = lat, lng
            break
            
    # Beri sedikit variasi spasial (jitter ~50-150 meter) berdasarkan index unik
    np.random.seed(int(row.get('No.', 1)) + 42)
    jitter_lat = np.random.uniform(-0.0020, 0.0020)
    jitter_lng = np.random.uniform(-0.0020, 0.0020)
    
    return pd.Series([base_lat + jitter_lat, base_lng + jitter_lng], index=['lat', 'lng'])

# Terapkan fungsi untuk menghasilkan kolom lat dan lng
df[['latitude', 'longitude']] = df.apply(assign_coordinates, axis=1)

print("Koordinat berhasil dibuat untuk semua kos!")
df[['Nama Kos', 'Alamat', 'latitude', 'longitude']].head()

Koordinat berhasil dibuat untuk semua kos!


,Nama Kos,Alamat,latitude,longitude
0,Kost Putri Asri,"Puluhdadi No.416, Ngropoh, Caturtunggal, Kec. ...",-7.773040,110.408436
1,Kost Putri Pak Seno,"Jl. Seturan No.1, Kledokan, Caturtunggal, Kec....",-7.775161,110.407919
2,Kost Perum APH Seturan A.24,"Perum APH Seturan Blok A.24 Kledokan, Caturtun...",-7.774544,110.409698
3,Graha Seturan Kost Putri,"Jln. Seturan Raya, Kledokan, Caturtunggal, Dep...",-7.775365,110.410039
4,Kost Diajeng,"Puluhdadi, Kledokan, Caturtunggal, Kec. Depok,...",-7.778046,110.411398


In [7]:
def get_marker_color(harga):
    if pd.isna(harga):
        return 'gray'
    elif harga < 800000:
        return 'green'
    elif harga <= 1200000:
        return 'orange'
    else:
        return 'red'

# Tambahkan kolom warna ke dataframe
df['marker_color'] = df['harga_min'].apply(get_marker_color)
print("Distribusi Warna Pin Kos:")
print(df['marker_color'].value_counts())

Distribusi Warna Pin Kos:
marker_color
gray      33
orange    14
green     11
red        5
Name: count, dtype: int64


In [8]:
# 1. Inisialisasi Peta Folium (Pusat di Seturan)
map_center = [-7.7720, 110.4080]
kos_map = folium.Map(
    location=map_center,
    zoom_start=14,
    tiles="CartoDB positron"  # Tampilan peta modern, bersih, dan elegan
)

# 2. Landmark Kampus Sekitar SCBD (Seturan - Babarsari)
CAMPUS_LANDMARKS = [
    {"name": "STIE YKPN Yogyakarta", "coords": [-7.7755, 110.4078], "color": "darkblue", "icon": "graduation-cap"},
    {"name": "UPN 'Veteran' Yogyakarta (Kampus 1)", "coords": [-7.7618, 110.4086], "color": "darkblue", "icon": "graduation-cap"},
    {"name": "UAJY (Kampus 3 Babarsari)", "coords": [-7.7797, 110.4158], "color": "purple", "icon": "graduation-cap"}
]

for camp in CAMPUS_LANDMARKS:
    folium.Marker(
        location=camp["coords"],
        popup=folium.Popup(f"<b>🎓 {camp['name']}</b>", max_width=250),
        tooltip=f"Kampus: {camp['name']}",
        icon=folium.Icon(color=camp["color"], icon=camp["icon"], prefix="fa")
    ).add_to(kos_map)

# 3. Tambahkan Pin untuk Setiap Kos
for _, row in df.iterrows():
    # Format teks harga untuk popup
    if pd.notna(row['harga_min']):
        harga_label = f"Rp {int(row['harga_min']):,} / bulan".replace(",", ".")
    else:
        harga_label = "Tanya Pemilik"
        
    # Badges Fasilitas Singkat
    fac_badges = ""
    if row.get('has_ac'): fac_badges += "❄️ AC "
    if row.get('has_km_dalam'): fac_badges += "🚿 KM Dalam "
    if row.get('has_wifi'): fac_badges += "📶 Wi-Fi "
    if not fac_badges: fac_badges = "Fasilitas standar"

C:\Users\Candra\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\folium\raster_layers.py:129: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [9]:
# Simpan file peta HTML interaktif
output_html = "sebaran_kos_map.html"
kos_map.save(output_html)
print(f"Peta berhasil diekspor ke: {output_html}")

Peta berhasil diekspor ke: sebaran_kos_map.html
